# 03 feature engineering

Notebook khởi tạo cho dự án UAV-FD.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_uav = pd.read_csv(
    "../data/processed/imu_combined.csv"
)

print("Kích thước dữ liệu:")
print(df_uav.shape)

print("\n5 dòng đầu:")
print(df_uav.head())


Kích thước dữ liệu:
(767763, 9)

5 dòng đầu:
       TimeUS      AccX      AccY      AccZ      GyrX      GyrY      GyrZ  \
0  15037020.0 -0.228946  0.251855 -9.755047  0.002102 -0.003518 -0.002962   
1  15039579.0 -0.230974  0.247798 -9.754610  0.001381 -0.002813 -0.001937   
2  15071017.0 -0.236723  0.239819 -9.790745  0.001758  0.000215  0.000723   
3  15073159.0 -0.234398  0.239244 -9.795955  0.000414 -0.000258  0.000247   
4  15090845.0 -0.222734  0.259719 -9.768929  0.003893  0.001428  0.000665   

     source_file  Label  
0  NO_FAULT1.mat      0  
1  NO_FAULT1.mat      0  
2  NO_FAULT1.mat      0  
3  NO_FAULT1.mat      0  
4  NO_FAULT1.mat      0  


In [2]:

features = [
    "AccX", "AccY", "AccZ",
    "GyrX", "GyrY", "GyrZ"
]

print("Giá trị thiếu của từng kênh:")

print(df_uav[features].isna().sum())

print("\nKiểu dữ liệu:")

print(df_uav[features + ["TimeUS"]].dtypes)


Giá trị thiếu của từng kênh:
AccX    0
AccY    0
AccZ    0
GyrX    0
GyrY    0
GyrZ    0
dtype: int64

Kiểu dữ liệu:
AccX      float64
AccY      float64
AccZ      float64
GyrX      float64
GyrY      float64
GyrZ      float64
TimeUS    float64
dtype: object


In [3]:

# Chọn một file Normal để kiểm tra
file_name = df_uav.loc[
    df_uav["Label"] == 0, "source_file"
].iloc[0]

data_file = df_uav[
    df_uav["source_file"] == file_name
].copy()

# TimeUS là thời gian tính bằng microsecond
time_diff = data_file["TimeUS"].diff().dropna()

print("Tên file:", file_name)

print("\nKhoảng thời gian giữa các mẫu:")
print(time_diff.describe())

print("\nMedian khoảng cách mẫu (microsecond):")
print(time_diff.median())

# Ước tính tần số lấy mẫu
positive_diff = time_diff[time_diff > 0]

if not positive_diff.empty:
    sampling_rate = 1_000_000 / positive_diff.median()
    print("\nTần số lấy mẫu ước tính (Hz):")
    print(round(sampling_rate, 2))


Tên file: NO_FAULT1.mat

Khoảng thời gian giữa các mẫu:
count    26161.000000
mean      2756.393486
std       1271.245900
min       1066.000000
25%       2275.000000
50%       2581.000000
75%       2912.000000
max      69738.000000
Name: TimeUS, dtype: float64

Median khoảng cách mẫu (microsecond):
2581.0

Tần số lấy mẫu ước tính (Hz):
387.45


In [4]:

results = []

for file_name, data in df_uav.groupby("source_file", sort=False):

    # Tính khoảng cách thời gian giữa các dòng
    dt = data["TimeUS"].diff().dropna()

    # Chỉ dùng khoảng cách dương để ước tính tần số
    positive_dt = dt[dt > 0]

    if positive_dt.empty:
        sampling_rate = np.nan
        median_dt = np.nan
    else:
        median_dt = positive_dt.median()
        sampling_rate = 1_000_000 / median_dt

    # Đếm các khoảng thời gian không tăng
    non_positive = (dt <= 0).sum()

    # Đếm khoảng cách lớn hơn 5 lần median
    if pd.notna(median_dt):
        large_gaps = (dt > 5 * median_dt).sum()
    else:
        large_gaps = np.nan

    results.append({
        "File": file_name,
        "Label": data["Label"].iloc[0],
        "Samples": len(data),
        "Median_dt_us": median_dt,
        "Sampling_Rate_Hz": sampling_rate,
        "Non_Positive_dt": non_positive,
        "Large_Gaps": large_gaps
    })

sampling_summary = pd.DataFrame(results)

print(
    sampling_summary.round(2).to_string(index=False)
)


           File  Label  Samples  Median_dt_us  Sampling_Rate_Hz  Non_Positive_dt  Large_Gaps
  NO_FAULT1.mat      0    26162        2581.0            387.45                0          15
  NO_FAULT2.mat      0    57456        2619.0            381.83                0          21
  NO_FAULT3.mat      0    42977        2654.0            376.79                0          22
  NO_FAULT4.mat      0    36855        2629.0            380.37                0          19
  NO_FAULT5.mat      0    34790        2638.0            379.08                0          18
  NO_FAULT6.mat      0    37880        2643.0            378.36                0          16
FAULT_M1_10.mat      1    47019        2661.0            375.80                0          16
 FAULT_M1_5.mat      1    64276        2625.0            380.95                0          24
FAULT_M2_10.mat      1    53865        2653.0            376.93                0          13
 FAULT_M2_5.mat      1    39209        2638.0            379.08       

In [5]:

import pandas as pd
import numpy as np

WINDOW_SECONDS = 2
WINDOW_US = WINDOW_SECONDS * 1_000_000

all_windows = []

for file_name, data in df_uav.groupby("source_file", sort=False):

    data = data.sort_values("TimeUS").copy()

    # Tính thời gian tương đối từ đầu file
    start_time = data["TimeUS"].iloc[0]

    data["Relative_TimeUS"] = data["TimeUS"] - start_time

    # Xác định cửa sổ 2 giây
    data["Window_ID"] = (
        data["Relative_TimeUS"] // WINDOW_US
    ).astype(int)

    # Kiểm tra khoảng gián đoạn trong mỗi cửa sổ
    median_dt = data["TimeUS"].diff()
    median_dt = median_dt[median_dt > 0].median()

    for window_id, window_data in data.groupby("Window_ID"):

        # Bỏ cửa sổ có ít hơn 100 mẫu
        if len(window_data) < 100:
            continue

        # Kiểm tra khoảng cách giữa các mẫu trong cửa sổ
        dt = window_data["TimeUS"].diff().dropna()

        if (dt > 5 * median_dt).any():
            continue

        all_windows.append({
            "source_file": file_name,
            "Window_ID": window_id,
            "Label": window_data["Label"].iloc[0],
            "Samples": len(window_data),
            "Start_TimeUS": window_data["TimeUS"].iloc[0],
            "End_TimeUS": window_data["TimeUS"].iloc[-1]
        })

window_summary = pd.DataFrame(all_windows)

print("Tổng số cửa sổ hợp lệ:", len(window_summary))

print("\n5 cửa sổ đầu:")
print(window_summary.head())

print("\nSố cửa sổ theo Label:")
print(window_summary["Label"].value_counts())


Tổng số cửa sổ hợp lệ: 988

5 cửa sổ đầu:
     source_file  Window_ID  Label  Samples  Start_TimeUS  End_TimeUS
0  NO_FAULT1.mat          2      0      763    19037083.0  21036268.0
1  NO_FAULT1.mat          3      0      757    21039120.0  23036163.0
2  NO_FAULT1.mat          4      0      748    23038755.0  25030639.0
3  NO_FAULT1.mat          5      0      719    25037129.0  27032848.0
4  NO_FAULT1.mat          6      0      718    27040275.0  29035100.0

Số cửa sổ theo Label:
Label
1    689
0    299
Name: count, dtype: int64


In [6]:

print("=== KIEM TRA WINDOWING ===")

# 1. Số file còn lại sau Windowing
print("\nSố file theo Label:")

print(
    window_summary.groupby("Label")["source_file"].nunique()
)

# 2. Số cửa sổ của từng file
print("\nSố cửa sổ mỗi file:")

print(
    window_summary.groupby(
        ["source_file", "Label"]
    ).size()
)

# 3. Số mẫu trong mỗi cửa sổ
print("\nThống kê số mẫu:")

print(window_summary["Samples"].describe())

# 4. Kiểm tra cửa sổ trùng nhau
print("\nSố cửa sổ trùng ID trong cùng file:")

print(
    window_summary.duplicated(
        subset=["source_file", "Window_ID"]
    ).sum()
)


=== KIEM TRA WINDOWING ===

Số file theo Label:
Label
0     6
1    12
Name: source_file, dtype: int64

Số cửa sổ mỗi file:
source_file      Label
FAULT_M1_10.mat  1        64
FAULT_M1_5.mat   1        81
FAULT_M2_10.mat  1        74
FAULT_M2_5.mat   1        45
FAULT_M3_10.mat  1        52
FAULT_M3_5.mat   1        83
FAULT_M4_10.mat  1        48
FAULT_M4_5.mat   1        37
FAULT_M5_10.mat  1        59
FAULT_M5_5.mat   1        38
FAULT_M6_10.mat  1        65
FAULT_M6_5.mat   1        43
NO_FAULT1.mat    0        33
NO_FAULT2.mat    0        72
NO_FAULT3.mat    0        53
NO_FAULT4.mat    0        46
NO_FAULT5.mat    0        44
NO_FAULT6.mat    0        51
dtype: int64

Thống kê số mẫu:
count    988.000000
mean     699.079960
std       68.473045
min      103.000000
25%      688.000000
50%      691.000000
75%      737.250000
max      773.000000
Name: Samples, dtype: float64

Số cửa sổ trùng ID trong cùng file:
0


In [7]:

feature_rows = []

# 6 kênh cảm biến IMU
imu_columns = [
    "AccX", "AccY", "AccZ",
    "GyrX", "GyrY", "GyrZ"
]

# Chỉ lấy những cửa sổ đã được giữ lại
for row in window_summary.itertuples(index=False):

    window_data = df_uav[
        (df_uav["source_file"] == row.source_file)
        & (df_uav["TimeUS"] >= row.Start_TimeUS)
        & (df_uav["TimeUS"] <= row.End_TimeUS)
    ]

    feature_dict = {
        "source_file": row.source_file,
        "Window_ID": row.Window_ID,
        "Label": row.Label
    }

    for col in imu_columns:

        signal = window_data[col]

        feature_dict[f"{col}_Mean"] = signal.mean()
        feature_dict[f"{col}_Std"] = signal.std()
        feature_dict[f"{col}_Min"] = signal.min()
        feature_dict[f"{col}_Max"] = signal.max()

        feature_dict[f"{col}_RMS"] = np.sqrt(
            np.mean(signal**2)
        )

        feature_dict[f"{col}_IQR"] = (
            signal.quantile(0.75)
            - signal.quantile(0.25)
        )

    feature_rows.append(feature_dict)

df_features = pd.DataFrame(feature_rows)

print("Kích thước Feature Dataset:")
print(df_features.shape)

print("\n5 dòng đầu:")
print(df_features.head())


Kích thước Feature Dataset:
(988, 39)

5 dòng đầu:
     source_file  Window_ID  Label  AccX_Mean  AccX_Std  AccX_Min  AccX_Max  \
0  NO_FAULT1.mat          2      0  -0.222921  0.089518 -0.461586  0.054663   
1  NO_FAULT1.mat          3      0  -0.219430  0.151110 -0.699179  0.521897   
2  NO_FAULT1.mat          4      0  -0.100556  0.204210 -0.750983  0.520922   
3  NO_FAULT1.mat          5      0  -0.329185  0.251941 -0.981678  0.395152   
4  NO_FAULT1.mat          6      0  -0.326704  0.233957 -0.901447  0.392033   

   AccX_RMS  AccX_IQR  AccY_Mean  ...  GyrY_Min  GyrY_Max  GyrY_RMS  GyrY_IQR  \
0  0.240202  0.113170   0.243814  ... -0.109844  0.107947  0.038137  0.057485   
1  0.266371  0.178987   0.255869  ... -0.205302  0.222684  0.061158  0.073517   
2  0.227502  0.270293   0.596778  ... -0.347959  0.224747  0.095305  0.107871   
3  0.414426  0.341089   0.829775  ... -0.570146  0.568357  0.209733  0.239296   
4  0.401740  0.294751   1.434939  ... -0.500522  0.307709  0.133553  

In [8]:

print("Số giá trị NaN:")
print(df_features.isna().sum().sum())

print("\nSố giá trị vô hạn:")
numeric_cols = df_features.select_dtypes(
    include="number"
).columns

print(
    np.isinf(df_features[numeric_cols].to_numpy()).sum()
)

print("\nSố file còn trong Feature Dataset:")
print(
    df_features.groupby("Label")["source_file"].nunique()
)

print("\nSố cửa sổ theo Label:")
print(df_features["Label"].value_counts())


Số giá trị NaN:
0

Số giá trị vô hạn:
0

Số file còn trong Feature Dataset:
Label
0     6
1    12
Name: source_file, dtype: int64

Số cửa sổ theo Label:
Label
1    689
0    299
Name: count, dtype: int64


In [9]:
print(df_features.shape)

(988, 39)


In [10]:

# Lưu bảng đặc trưng
df_features.to_csv(
    "../data/processed/imu_features.csv",
    index=False
)

print("Đã lưu Feature Dataset!")

print("Kích thước:", df_features.shape)


Đã lưu Feature Dataset!
Kích thước: (988, 39)
